# CyFinX — FAA Investigation Runner

This notebook runs the integrated FAA workflow using Gemma 4 on a Google Colab GPU.

## Before running

1. Connect this notebook to a Colab GPU runtime (T4 is supported).
2. Make the current `app/` folder from the `faa-implementation` branch available to Colab at `/content/app` using the VS Code Colab connection.
3. Keep the operational database in Google Drive at `My Drive/CyFinX/database/cyfinx_operational_v1.db`.

Run the notebook from top to bottom after each runtime restart. GitHub remains the permanent source-code store, Google Drive holds the permanent database and investigation outputs, and Colab's local runtime is used only for execution.


In [2]:

from pathlib import Path
from google.colab import drive
import os
import shutil
import subprocess
import sqlite3
import sys

# --------------------------------------------------
# 1. Project configuration
# --------------------------------------------------

REPO_URL = "https://github.com/hibaaqy/CyFinX-implementation.git"
BRANCH = "faa-implementation"

PROJECT_DIR = Path("/content/CyFinX")

DRIVE_DB = Path(
    "/content/drive/MyDrive/CyFinX/database/cyfinx_operational_v1.db"
)

LOCAL_DB = PROJECT_DIR / "data/database/cyfinx_operational_v1.db"

DRIVE_RESULTS = Path("/content/drive/MyDrive/CyFinX/results")


# --------------------------------------------------
# 2. Mount Google Drive
# --------------------------------------------------

drive.mount("/content/drive")


# --------------------------------------------------
# 3. Clone or update the GitHub repository
# --------------------------------------------------

def git(*args):
    subprocess.run(["git", *args], check=True)

if (PROJECT_DIR / ".git").is_dir():

    print("Existing repository found. Updating code...")

    git("-C", str(PROJECT_DIR), "fetch", "origin", BRANCH)

    git(
        "-C", str(PROJECT_DIR),
        "checkout", "-B", BRANCH, f"origin/{BRANCH}"
    )

else:

    if PROJECT_DIR.exists():
        backup = Path("/content/CyFinX_previous")
        i = 1

        while backup.exists():
            backup = Path(f"/content/CyFinX_previous_{i}")
            i += 1

        PROJECT_DIR.rename(backup)
        print(f"Previous folder moved to: {backup}")

    print("Cloning CyFinX from GitHub...")

    git(
        "clone",
        "--single-branch",
        "--branch", BRANCH,
        "--depth", "1",
        REPO_URL,
        str(PROJECT_DIR)
    )


# --------------------------------------------------
# 4. Verify project files
# --------------------------------------------------

if not (PROJECT_DIR / "app/main.py").is_file():
    raise FileNotFoundError("app/main.py not found in repository.")

print("CyFinX source code is ready.")


# --------------------------------------------------
# 5. Copy database from Google Drive
# --------------------------------------------------

if not DRIVE_DB.is_file():
    raise FileNotFoundError(
        f"Database not found in Google Drive: {DRIVE_DB}"
    )

LOCAL_DB.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(DRIVE_DB, LOCAL_DB)

print("Database copied successfully.")


# --------------------------------------------------
# 6. Set database path BEFORE importing CyFinX
# --------------------------------------------------

os.environ["CYFINX_OPERATIONAL_DB"] = str(LOCAL_DB)

with sqlite3.connect(LOCAL_DB.as_uri() + "?mode=ro", uri=True) as conn:
    integrity = conn.execute("PRAGMA integrity_check").fetchone()[0]

if integrity != "ok":
    raise RuntimeError(f"Database integrity check failed: {integrity}")

print("Database integrity:", integrity)


# --------------------------------------------------
# 7. Configure Python project path
# --------------------------------------------------

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


# --------------------------------------------------
# 8. Prepare persistent results folder
# --------------------------------------------------

DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# 9. Display environment information
# --------------------------------------------------

commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    text=True
).strip()

print("\n========== CYFINX ENVIRONMENT READY ==========")
print("Project:", PROJECT_DIR)
print("Branch:", BRANCH)
print("Git commit:", commit)
print("Database:", LOCAL_DB)
print("Results folder:", DRIVE_RESULTS)

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch has not been installed yet.")

print("==============================================")


Mounted at /content/drive
Previous folder moved to: /content/CyFinX_previous
Cloning CyFinX from GitHub...
CyFinX source code is ready.
Database copied successfully.
Database integrity: ok

========== CYFINX ENVIRONMENT READY ==========
Project: /content/CyFinX
Branch: faa-implementation
Git commit: a430f46
Database: /content/CyFinX/data/database/cyfinx_operational_v1.db
Results folder: /content/drive/MyDrive/CyFinX/results
CUDA available: False


In [3]:
from pathlib import Path, PurePosixPath
import os
import sys
import shutil
import zipfile
import torch

PROJECT_DIR = Path('/content/CyFinX')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_APP_DIR = PROJECT_DIR / 'app'

UPDATE_ZIP = Path('/content/CyFinX_Gemma_Performance_Update.zip')
APP_ZIP = Path('/content/app.zip')


def extract_archive_normalized(archive_path, destination_root, allowed_roots=None):
    """Extract ZIP entries safely, converting Windows backslashes to '/' paths."""
    extracted = 0
    with zipfile.ZipFile(archive_path, 'r') as archive:
        for info in archive.infolist():
            normalized = info.filename.replace('\\', '/')
            parts = PurePosixPath(normalized).parts
            if not parts or normalized.startswith('/') or '..' in parts:
                continue
            if allowed_roots and parts[0] not in allowed_roots:
                continue
            destination = destination_root.joinpath(*parts)
            if info.is_dir() or normalized.endswith('/'):
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(info))
                extracted += 1
    return extracted


# Prefer the updated package (app + benchmark runner + tests).
if UPDATE_ZIP.is_file():
    count = extract_archive_normalized(
        UPDATE_ZIP, PROJECT_DIR, allowed_roots={'app', 'scripts', 'tests'}
    )
    print(f'Extracted updated CyFinX package: {count} files')
elif APP_ZIP.is_file():
    count = extract_archive_normalized(APP_ZIP, PROJECT_DIR, allowed_roots={'app'})
    print(f'Extracted app archive: {count} files')
elif (Path('/content/app') / 'main.py').is_file():
    shutil.copytree(Path('/content/app'), TARGET_APP_DIR, dirs_exist_ok=True)
else:
    # Recover files previously extracted by Windows ZIP tools as flat names
    # such as "app\\main.py" in /content.
    recovered = 0
    for item in list(Path('/content').iterdir()):
        if not item.name.startswith('app\\'):
            continue
        relative = PurePosixPath(item.name.replace('\\', '/')).parts[1:]
        if not relative or '..' in relative:
            continue
        destination = TARGET_APP_DIR.joinpath(*relative)
        if item.is_dir():
            shutil.copytree(item, destination, dirs_exist_ok=True)
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, destination)
        recovered += 1
    print(f'Recovered flat app entries: {recovered}')

if not (TARGET_APP_DIR / 'main.py').is_file():
    raise FileNotFoundError(
        'CyFinX app source is still missing. Upload '
        'CyFinX_Gemma_Performance_Update.zip to /content, then rerun this cell.'
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)

app_file_count = sum(1 for item in TARGET_APP_DIR.rglob('*') if item.is_file())
print('Project directory:', PROJECT_DIR)
print('App source:', TARGET_APP_DIR / 'main.py')
print('App files:', app_file_count)
print('Python:', sys.executable)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('A Colab GPU runtime is required. Select a T4 GPU runtime.')
print('GPU:', torch.cuda.get_device_name(0))


Recovered flat app entries: 0
Project directory: /content/CyFinX
App source: /content/CyFinX/app/main.py
App files: 18
Python: /usr/bin/python3
CUDA available: False


RuntimeError: A Colab GPU runtime is required. Select a T4 GPU runtime.

## Install application dependencies

Colab already provides a CUDA-enabled PyTorch build. The next cell installs the remaining CyFinX runtime packages without replacing Colab's PyTorch/CUDA installation.


In [ ]:
%pip install -q "SQLAlchemy>=2.0,<3.0" "pandas>=2.2,<4.0" "matplotlib>=3.8,<4.0" "Pillow>=10" "transformers>=5.0" "accelerate>=1.0" "huggingface_hub>=0.30"


## Connect the permanent Google Drive database

The master SQLite database stays in Google Drive. Each Colab runtime receives a local working copy for reliable, fast SQLite access. The application is configured to use that local copy **before any `app` modules are imported**. The Drive master is not modified by this notebook.


In [ ]:
from google.colab import drive
import sqlite3

# Mount the permanent Google Drive storage.
drive.mount('/content/drive')

DRIVE_DB_PATH = Path(
    '/content/drive/MyDrive/CyFinX/database/cyfinx_operational_v1.db'
)
if not DRIVE_DB_PATH.is_file():
    raise FileNotFoundError(
        'The permanent database was not found. Expected it at: '
        f'{DRIVE_DB_PATH}'
    )

# Copy the Drive master into the current Colab runtime.
LOCAL_DB_PATH = PROJECT_DIR / 'data' / 'database' / 'cyfinx_operational_v1.db'
LOCAL_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_DB_PATH, LOCAL_DB_PATH)

if LOCAL_DB_PATH.stat().st_size != DRIVE_DB_PATH.stat().st_size:
    raise IOError('Database copy size does not match the Google Drive master file.')

# Check the copied SQLite database before the application opens it.
with sqlite3.connect(LOCAL_DB_PATH.resolve().as_uri() + '?mode=ro', uri=True) as connection:
    integrity_result = connection.execute('PRAGMA quick_check').fetchone()[0]
if integrity_result != 'ok':
    raise RuntimeError(f'SQLite integrity check failed: {integrity_result}')

# IMPORTANT: set this before importing app.main or any app database modules.
os.environ['CYFINX_OPERATIONAL_DB'] = str(LOCAL_DB_PATH)

print('Permanent Drive database:', DRIVE_DB_PATH)
print('Colab working copy:', LOCAL_DB_PATH)
print('Database size:', f'{LOCAL_DB_PATH.stat().st_size:,} bytes')
print('SQLite quick_check:', integrity_result)
print('CYFINX_OPERATIONAL_DB configured.')


## Load Gemma 4

The model is loaded once and passed to the FAA, report generator, vision agent, and detective agent. This avoids loading multiple model copies into GPU memory.


In [ ]:
from app.llm.gemma_client import GemmaClient

gemma = GemmaClient.load()
print('Gemma 4 is ready.')


## Run one FAA investigation

The FAA dynamically repeats **Planning → Information-Gathering → Analysis**. The FAA itself decides when to stop. `MAX_STEPS` is only a safety cap; reaching it does not force a final report or a fraud/legitimate decision.

After the run, this cell saves the investigation JSON, reports, and any chart artifacts to Google Drive so they remain available after the Colab runtime ends.


In [ ]:
from app.main import run_investigation
from datetime import datetime
import json
import re

ALERT_ID = 'ALT0000089'
MAX_STEPS = 12  # Runtime safety cap only; FAA Analysis decides whether to stop.
RUN_EVIDENCE_QUALITY_EVALUATION = False  # Optional FAA paper evaluation.

result = run_investigation(
    alert_id=ALERT_ID,
    max_steps=MAX_STEPS,
    gemma_client=gemma,
    evaluate_evidence=RUN_EVIDENCE_QUALITY_EVALUATION,
)

print('Investigation status:', result['status'])
print('Investigation steps:', len(result.get('steps', [])))
if result['status'] == 'completed':
    print('\nFINAL RGA REPORT\n')
    print(result['reports']['final_report'])
    print('\nDETECTIVE DECISION\n')
    print(json.dumps(result['detective_decision'], indent=2, ensure_ascii=False))
else:
    print(result.get('message', 'Investigation paused without a final decision.'))

performance = result.get('performance_metrics', {})
print('\nGEMMA PERFORMANCE')
for key in (
    'model_call_count', 'successful_model_calls', 'failed_model_calls',
    'total_input_tokens', 'total_output_tokens',
    'total_model_inference_seconds', 'average_call_latency_seconds',
    'overall_output_tokens_per_second', 'framework_wall_time_seconds',
    'investigation_step_count', 'information_gathering_execution_seconds',
    'code_generation_retry_count', 'vision_agent_invocations',
    'run_peak_gpu_memory_allocated_gib', 'run_peak_gpu_memory_reserved_gib',
):
    print(f'{key}: {performance.get(key)}')
print('\nCALLS BY FAA STAGE')
for stage, values in performance.get('calls_by_stage', {}).items():
    print(stage, '->', values)

print('\nColab result file:', result.get('saved_result_path'))

# Permanently archive this run in Google Drive.
safe_alert_id = re.sub(r'[^A-Za-z0-9_.-]', '_', ALERT_ID)
run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
DRIVE_RESULTS_DIR = (
    Path('/content/drive/MyDrive/CyFinX/investigation_results')
    / f'{safe_alert_id}_{run_timestamp}'
)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=False)

saved_result_path = result.get('saved_result_path')
if saved_result_path and Path(saved_result_path).is_file():
    shutil.copy2(saved_result_path, DRIVE_RESULTS_DIR / 'investigation_result.json')
if result['status'] == 'completed':
    reports = result.get('reports', {})
    (DRIVE_RESULTS_DIR / 'final_report.md').write_text(
        reports.get('final_report', ''), encoding='utf-8'
    )
    (DRIVE_RESULTS_DIR / 'unfiltered_report.md').write_text(
        reports.get('unfiltered_report', ''), encoding='utf-8'
    )
    (DRIVE_RESULTS_DIR / 'detective_decision.json').write_text(
        json.dumps(result.get('detective_decision', {}), indent=2, ensure_ascii=False),
        encoding='utf-8',
    )
    (DRIVE_RESULTS_DIR / 'performance_metrics.json').write_text(
        json.dumps(performance, indent=2, ensure_ascii=False), encoding='utf-8'
    )

LOCAL_STEP_ARTIFACTS = PROJECT_DIR / 'reports' / 'step_artifacts' / safe_alert_id
if LOCAL_STEP_ARTIFACTS.is_dir():
    shutil.copytree(
        LOCAL_STEP_ARTIFACTS,
        DRIVE_RESULTS_DIR / 'step_artifacts',
        dirs_exist_ok=True,
    )
print('Permanent Google Drive archive:', DRIVE_RESULTS_DIR)


## Preview available alerts for the 10-case framework benchmark

This query reads only the operational `Fraud_Alerts` and `Transactions` tables. It does not retrieve or expose Ground_Truth to Gemma. Choose the same 10 Alert_ID values for any later Qwen full-framework comparison.


In [ ]:
import pandas as pd

with sqlite3.connect(LOCAL_DB_PATH.resolve().as_uri() + '?mode=ro', uri=True) as connection:
    alerts_preview = pd.read_sql_query(
        """
        SELECT
            a.Alert_ID,
            a.Alert_type,
            a.Criticality_level,
            a.Criticality_score,
            a.Alert_timestamp,
            t.Transaction_Type,
            t.Amount,
            t.Channel
        FROM Fraud_Alerts AS a
        LEFT JOIN Transactions AS t
            ON t.Transaction_ID = a.Transaction_ID
        ORDER BY a.Alert_timestamp, a.Alert_ID
        """,
        connection,
    )

display(alerts_preview.head(30))
print('Total alert records available:', len(alerts_preview))


## Run the 10-alert Gemma benchmark

Run this only after the single-alert investigation succeeds. Replace the empty list with exactly 10 Alert_ID values from the preview. The benchmark reuses the already-loaded Gemma model, runs the complete FAA framework for each alert, and writes per-alert JSON plus a summary CSV. A case paused at the FAA safety cap is reported as paused, not as a final decision.


In [ ]:
from scripts.benchmark_gemma_framework import run_benchmark

ALERT_IDS = [
    # Paste exactly 10 Alert_ID values from alerts_preview here.
]

if len(ALERT_IDS) != 10:
    raise ValueError(f'Please provide exactly 10 Alert_ID values; received {len(ALERT_IDS)}.')

BENCHMARK_OUTPUT_DIR = Path(
    '/content/drive/MyDrive/CyFinX/benchmarks/gemma_faa_10_alerts'
)

benchmark = run_benchmark(
    alert_ids=ALERT_IDS,
    max_steps=12,
    gemma_client=gemma,
    evaluate_evidence=False,
    output_dir=BENCHMARK_OUTPUT_DIR,
    resume=True,  # Re-running skips cases already completed in this folder.
)

print('Benchmark CSV:', benchmark['summary_csv'])
print('Benchmark archive:', benchmark['output_dir'])
